# Metadata Extraction — ExamCard and PAR Files

Shows every parameter that feeds into the BIDS JSON sidecars, where it comes from, and
what the corresponding BIDS field name is.

| Sequence | BIDS modality | ExamCard section | PAR file |
|---|---|---|---|
| T1 TFE with IR prepulse | `anat/T1w` | `T1_NFB` | `*wipt1_nfb.par` |
| 64-dir DTI EPI | `dwi/dwi` | `q64 wb` | `*wipq64wb.par` |

**Source hierarchy (later sources override earlier ones):**
1. ExamCard — static protocol-level parameters
2. PAR header — session-level values confirmed by the scanner
3. PAR image table — per-subject rescaling factors and timing

In [1]:
from pathlib import Path
import xml.etree.ElementTree as Et
import examcard as ec
import examcard.examcard as ecm
import pandas as pd

EXAMCARD_PATH = Path("/Users/hugofluhr/phd_local/data/cocaine_study/mrs_bluko_099_20260522_BLUKO_20240321_old_waterSup.ExamCard")
T1_PAR_PATH   = Path("/Users/hugofluhr/phd_local/data/cocaine_study/DTI/002_bluko_mrs_20240905_111413_3_1_wipt1_nfb.par")
DWI_PAR_PATH  = Path("/Users/hugofluhr/phd_local/data/cocaine_study/DTI/002_bluko_mrs_20240905_112851_5_1_wipq64wb.par")

In [2]:
import re

# ── ExamCard ─────────────────────────────────────────────────────────────────
raw = EXAMCARD_PATH.read_bytes()
end = raw.rfind(b'</SOAP-ENV:Envelope>') + len(b'</SOAP-ENV:Envelope>')
ecm.Et.parse = lambda f, *a, **kw: Et.ElementTree(Et.fromstring(raw[:end]))
parsed_ec = ec.parse(EXAMCARD_PATH)


def resolve_param(seq_name, key):
    """Return (raw_value, human_readable_string) for one ExamCard parameter."""
    seq    = parsed_ec[seq_name]
    params = seq.get('protocolParameter', {})
    enums  = seq.get('enumDescriptions', [])
    emap   = seq.get('enumMap', {})
    v = params.get(key)
    if v is None:
        return None, None
    idx = emap.get(key)
    if idx is not None:
        try:
            return v, enums[idx].values[v]
        except Exception:
            pass
    return v, None


# ── PAR helpers ──────────────────────────────────────────────────────────────
def parse_par_header(par_path):
    """Return {key: value} dict of PAR dot-prefixed header lines."""
    hdr = {}
    with open(par_path, encoding='latin-1') as f:
        for line in f:
            if line.startswith('.') and ':' in line:
                key, _, val = line[1:].partition(':')
                hdr[key.strip()] = val.strip()
    return hdr


def parse_par_image_table(par_path):
    """Return list of tokenised rows from the PAR image table."""
    rows, in_table = [], False
    with open(par_path, encoding='latin-1') as f:
        for line in f:
            stripped = line.strip()
            if 'IMAGE INFORMATION =' in stripped:
                in_table = True
            elif in_table and stripped and not stripped.startswith('#'):
                cols = stripped.split()
                if len(cols) > 35:
                    rows.append(cols)
    return rows


t1_hdr   = parse_par_header(T1_PAR_PATH)
t1_rows  = parse_par_image_table(T1_PAR_PATH)
dwi_hdr  = parse_par_header(DWI_PAR_PATH)
dwi_rows = parse_par_image_table(DWI_PAR_PATH)

print(f"T1  PAR: {len(t1_rows)} image-table rows")
print(f"DWI PAR: {len(dwi_rows)} image-table rows")

# ── Sidecar field sets (actual content of sub-*/anat/*.json and sub-*/dwi/*.json) ──
T1_SIDECAR_FIELDS = {
    'Manufacturer', 'ProtocolName', 'SeriesNumber', 'AcquisitionNumber',
    'PhilipsRescaleSlope', 'PhilipsRescaleIntercept', 'PhilipsScaleSlope',
    'EchoTime', 'RepetitionTime', 'ImageOrientationPatientDICOM',
}

DWI_SIDECAR_FIELDS = {
    'Manufacturer', 'ManufacturersModelName', 'MagneticFieldStrength',
    'ProtocolName', 'SeriesNumber', 'AcquisitionNumber',
    'PhilipsRescaleSlope', 'PhilipsRescaleIntercept', 'PhilipsScaleSlope',
    'EchoTime', 'ImageOrientationPatientDICOM', 'RepetitionTime', 'FlipAngle',
    'PhaseEncodingDirection', 'TotalReadoutTime',
    'SliceThickness', 'NumberOfSlices', 'ReconMatrixPE',
    'ParallelReductionFactorInPlane', 'ParallelAcquisitionTechnique',
    'MultibandAccelerationFactor', 'ReceiveCoilName', 'InstitutionName',
}


def in_sidecar(bids_field_str, sidecar_fields):
    """Return '✓' if any BIDS field in the string is present in sidecar_fields."""
    if bids_field_str == '—':
        return '✗'
    # strip unit annotations like ' (÷1000)', split comma-separated names
    names = [re.sub(r'\s*\(.*?\)', '', f).strip() for f in bids_field_str.split(',')]
    return '✓' if any(n in sidecar_fields for n in names) else '✗'

T1  PAR: 160 image-table rows
DWI PAR: 3900 image-table rows


## T1w — `T1_NFB` + `*wipt1_nfb.par`

3D TFE (Turbo Field Echo) with inversion prepulse — Philips equivalent of MP-RAGE.

In [3]:
# ── ExamCard rows ─────────────────────────────────────────────────────────────
T1_EC_FIELDS = [
    ('EX_ACQ_scan_mode',             'Acquisition type',                  'MRAcquisitionType'),
    ('EX_ACQ_fast_imaging_mode',     'Fast imaging mode',                 '—'),
    ('EX_ACQ_flip_angle',            'Flip angle [°]',                    'FlipAngle'),
    ('EX_ACQ_first_echo_time',       'Echo time [ms]',                    'EchoTime (÷1000)'),
    ('EX_TFE_factor',                'TFE turbo factor (shots per TI)',   '—'),
    ('EX_TFE_interval',              'TFE shot interval / TR [ms]',       'RepetitionTime (÷1000)'),
    ('EX_TFEPP_prepulses',           'Prepulse type',                     '—'),
    ('EX_TFEPP_pre_T1_delay',        'Inversion delay / TI [ms]',         'InversionTime (÷1000)'),
    ('EX_GEO_acq_slice_thickness',   'Slice thickness [mm]',              'SliceThickness'),
    ('EX_GEO_sense_enable',          'SENSE enabled',                     '—'),
    ('EX_GEO_sense_p_red_factor',    'SENSE phase reduction factor',      'ParallelReductionFactorInPlane'),
    ('EX_GEO_sense_s_red_factor',    'SENSE slice reduction factor',      'ParallelReductionFactorOutOfPlane'),
    ('EX_GEO_stacks_clinical_modes', 'Receive coil',                      'ReceiveCoilName'),
]

# ── PAR header rows ───────────────────────────────────────────────────────────
T1_HDR_FIELDS = [
    ('Acquisition nr',                  'Acquisition / series number',    'AcquisitionNumber, SeriesNumber'),
    ('Protocol name',                   'Protocol name',                  'ProtocolName'),
    ('Scan mode',                       'Scan mode',                      'MRAcquisitionType'),
    ('Technique',                       'Imaging technique',              '—'),
    ('Repetition time [msec]',          'Repetition time [ms]',           'RepetitionTime (÷1000)'),
    ('Max. number of slices/locations', 'Number of slices',               'NumberOfSlices'),
]

# ── PAR image-table columns (0-based) ─────────────────────────────────────────
T1_IMG_FIELDS = [
    (11, 'rescale intercept (RI)',    'PhilipsRescaleIntercept'),
    (12, 'rescale slope (RS)',        'PhilipsRescaleSlope'),
    (13, 'scale slope (SS)',          'PhilipsScaleSlope'),
    (22, 'slice thickness [mm]',      'SliceThickness'),
    (30, 'echo time [ms]',            'EchoTime (÷1000)'),
    (35, 'flip angle [°]',            'FlipAngle'),
    (39, 'TFE / turbo factor',        '—'),
    (40, 'inversion delay [ms]',      'InversionTime (÷1000)'),
]

# ── Build table ───────────────────────────────────────────────────────────────
t1_table_rows = []

for param, hr, bids in T1_EC_FIELDS:
    raw, resolved = resolve_param('T1_NFB', param)
    t1_table_rows.append(dict(
        param=param, human_readable=hr, bids_field=bids,
        value=(resolved if resolved is not None else raw),
        in_sidecar=in_sidecar(bids, T1_SIDECAR_FIELDS),
        source='ExamCard: T1_NFB',
    ))

for param, hr, bids in T1_HDR_FIELDS:
    t1_table_rows.append(dict(
        param=param, human_readable=hr, bids_field=bids,
        value=t1_hdr.get(param, '—'),
        in_sidecar=in_sidecar(bids, T1_SIDECAR_FIELDS),
        source='PAR header',
    ))

if t1_rows:
    first = t1_rows[0]
    for col, hr, bids in T1_IMG_FIELDS:
        vals = {r[col] for r in t1_rows}
        flag = '  ⚠ varies' if len(vals) > 1 else ''
        t1_table_rows.append(dict(
            param=f'col_{col}', human_readable=hr, bids_field=bids,
            value=f"{first[col]}{flag}",
            in_sidecar=in_sidecar(bids, T1_SIDECAR_FIELDS),
            source='PAR image table',
        ))

pd.set_option('display.max_colwidth', 60)
pd.DataFrame(t1_table_rows)

,param,human_readable,bids_field,value,in_sidecar,source
0,EX_ACQ_scan_mode,Acquisition type,MRAcquisitionType,3D,✗,ExamCard: T1_NFB
1,EX_ACQ_fast_imaging_mode,Fast imaging mode,—,TFE,✗,ExamCard: T1_NFB
2,EX_ACQ_flip_angle,Flip angle [°],FlipAngle,8.0,✗,ExamCard: T1_NFB
3,EX_ACQ_first_echo_time,Echo time [ms],EchoTime (÷1000),4.605638,✓,ExamCard: T1_NFB
4,EX_TFE_factor,TFE turbo factor (shots per TI),—,120,✗,ExamCard: T1_NFB
5,EX_TFE_interval,TFE shot interval / TR [ms],RepetitionTime (÷1000),3000.0,✓,ExamCard: T1_NFB
6,EX_TFEPP_prepulses,Prepulse type,—,INV,✗,ExamCard: T1_NFB
7,EX_TFEPP_pre_T1_delay,Inversion delay / TI [ms],InversionTime (÷1000),1000.0,✗,ExamCard: T1_NFB
8,EX_GEO_acq_slice_thickness,Slice thickness [mm],SliceThickness,2.0,✗,ExamCard: T1_NFB
9,EX_GEO_sense_enable,SENSE enabled,—,BLAST,✗,ExamCard: T1_NFB


## DWI — `q64 wb` + `*wipq64wb.par`

2D multi-slice EPI spin echo, 64-direction DTI acquisition.

In [4]:
# ── ExamCard rows ─────────────────────────────────────────────────────────────
DWI_EC_FIELDS = [
    ('EX_ACQ_scan_mode',             'Acquisition type',               'MRAcquisitionType'),
    ('EX_ACQ_fast_imaging_mode',     'Fast imaging mode',              '—'),
    ('EX_ACQ_imaging_sequence',      'Base imaging sequence',          '—'),
    ('EX_ACQ_flip_angle',            'Flip angle [°]',                 'FlipAngle'),
    ('EX_ACQ_first_echo_time',       'Echo time [ms]',                 'EchoTime (÷1000)'),
    ('EX_ACQ_repetition_time',       'Repetition time [ms]',           'RepetitionTime (÷1000)'),
    ('EX_GEO_acq_slice_thickness',   'Slice thickness [mm]',           'SliceThickness'),
    ('EX_GEO_sense_enable',          'SENSE enabled',                  '—'),
    ('EX_GEO_sense_p_red_factor',    'SENSE phase reduction factor',   'ParallelReductionFactorInPlane'),
    ('EX_GEO_stacks_clinical_modes', 'Receive coil',                   'ReceiveCoilName'),
    ('EX_DIFF_enable',               'Diffusion mode',                 '—'),
]

# ── PAR header rows ───────────────────────────────────────────────────────────
DWI_HDR_FIELDS = [
    ('Acquisition nr',                  'Acquisition / series number',     'AcquisitionNumber, SeriesNumber'),
    ('Protocol name',                   'Protocol name',                   'ProtocolName'),
    ('Scan mode',                       'Scan mode',                       'MRAcquisitionType'),
    ('Technique',                       'Imaging technique',               '—'),
    ('Repetition time [msec]',          'Repetition time [ms]',            'RepetitionTime (÷1000)'),
    ('Max. number of slices/locations', 'Number of slices',                'NumberOfSlices'),
    ('Max. number of dynamics',         'Number of volumes (b0 + dirs)',   '—'),
    ('Max. number of diffusion values', 'Number of unique b-values',       '—'),
    ('Max. number of gradient orients', 'Number of gradient directions',   '—'),
    ('EPI factor        <0,1=no EPI>',  'EPI factor',                      '—'),
]

# ── PAR image-table columns (0-based) ─────────────────────────────────────────
DWI_IMG_FIELDS = [
    (11, 'rescale intercept (RI)',    'PhilipsRescaleIntercept'),
    (12, 'rescale slope (RS)',        'PhilipsRescaleSlope'),
    (13, 'scale slope (SS)',          'PhilipsScaleSlope'),
    (22, 'slice thickness [mm]',      'SliceThickness'),
    (30, 'echo time [ms]',            'EchoTime (÷1000)'),
    (35, 'flip angle [°]',            'FlipAngle'),
]

# ── Build table ───────────────────────────────────────────────────────────────
dwi_table_rows = []

for param, hr, bids in DWI_EC_FIELDS:
    raw, resolved = resolve_param('q64 wb', param)
    dwi_table_rows.append(dict(
        param=param, human_readable=hr, bids_field=bids,
        value=(resolved if resolved is not None else raw),
        in_sidecar=in_sidecar(bids, DWI_SIDECAR_FIELDS),
        source='ExamCard: q64 wb',
    ))

for param, hr, bids in DWI_HDR_FIELDS:
    dwi_table_rows.append(dict(
        param=param, human_readable=hr, bids_field=bids,
        value=dwi_hdr.get(param, '—'),
        in_sidecar=in_sidecar(bids, DWI_SIDECAR_FIELDS),
        source='PAR header',
    ))

if dwi_rows:
    first = dwi_rows[0]
    for col, hr, bids in DWI_IMG_FIELDS:
        vals = {r[col] for r in dwi_rows}
        flag = '  ⚠ varies' if len(vals) > 1 else ''
        dwi_table_rows.append(dict(
            param=f'col_{col}', human_readable=hr, bids_field=bids,
            value=f"{first[col]}{flag}",
            in_sidecar=in_sidecar(bids, DWI_SIDECAR_FIELDS),
            source='PAR image table',
        ))

pd.DataFrame(dwi_table_rows)

,param,human_readable,bids_field,value,in_sidecar,source
0,EX_ACQ_scan_mode,Acquisition type,MRAcquisitionType,MS,✗,ExamCard: q64 wb
1,EX_ACQ_fast_imaging_mode,Fast imaging mode,—,EPI,✗,ExamCard: q64 wb
2,EX_ACQ_imaging_sequence,Base imaging sequence,—,SE,✗,ExamCard: q64 wb
3,EX_ACQ_flip_angle,Flip angle [°],FlipAngle,90.0,✓,ExamCard: q64 wb
4,EX_ACQ_first_echo_time,Echo time [ms],EchoTime (÷1000),50.8675,✓,ExamCard: q64 wb
5,EX_ACQ_repetition_time,Repetition time [ms],RepetitionTime (÷1000),None,✓,ExamCard: q64 wb
6,EX_GEO_acq_slice_thickness,Slice thickness [mm],SliceThickness,2.0,✓,ExamCard: q64 wb
7,EX_GEO_sense_enable,SENSE enabled,—,YES,✗,ExamCard: q64 wb
8,EX_GEO_sense_p_red_factor,SENSE phase reduction factor,ParallelReductionFactorInPlane,2.0,✓,ExamCard: q64 wb
9,EX_GEO_stacks_clinical_modes,Receive coil,ReceiveCoilName,32CH_HEAD_COIL,✓,ExamCard: q64 wb
